<a href="https://colab.research.google.com/github/kamalika-m6/Generative-ai-experiments/blob/exercise3/gen_ai_exp10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y datasets huggingface_hub transformers
!pip install -q "datasets==3.2.0" "huggingface_hub==0.27.1" "transformers==4.47.1" accelerate scikit-learn

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: huggingface_hub 1.23.0
Uninstalling huggingface_hub-1.23.0:
  Successfully uninstalled huggingface_hub-1.23.0
Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 107.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source

In [1]:
from datasets import load_dataset

print("Loading IMDB dataset...")

dataset = load_dataset("imdb")

print(dataset)

Loading IMDB dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [2]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import numpy as np
from sklearn.metrics import accuracy_score
import torch

# ---------------------------------------
# Check GPU
# ---------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)


# ---------------------------------------
# 1. Load IMDB dataset
# ---------------------------------------

print("\nLoading IMDB dataset...")

dataset = load_dataset("imdb")

small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test = dataset["test"].shuffle(seed=42).select(range(500))

print("Training samples:", len(small_train))
print("Testing samples:", len(small_test))


# ---------------------------------------
# 2. Load tokenizer
# ---------------------------------------

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)


# ---------------------------------------
# 3. Tokenize dataset
# ---------------------------------------

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


train_ds = small_train.map(
    tokenize,
    batched=True
)

test_ds = small_test.map(
    tokenize,
    batched=True
)


# Change label to labels
train_ds = train_ds.rename_column(
    "label",
    "labels"
)

test_ds = test_ds.rename_column(
    "label",
    "labels"
)


# ---------------------------------------
# 4. Load DistilBERT model
# ---------------------------------------

print("\nLoading DistilBERT model...")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)


# ---------------------------------------
# 5. Training arguments
# ---------------------------------------

args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    logging_steps=50,
    save_strategy="epoch",
    report_to="none"
)


# ---------------------------------------
# 6. Accuracy function
# ---------------------------------------

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    preds = np.argmax(
        predictions,
        axis=1
    )

    accuracy = accuracy_score(
        labels,
        preds
    )

    return {
        "accuracy": accuracy
    }


# ---------------------------------------
# 7. Create Trainer
# ---------------------------------------

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)


# ---------------------------------------
# 8. Train
# ---------------------------------------

print("\nStarting training...")

trainer.train()


# ---------------------------------------
# 9. Evaluate
# ---------------------------------------

print("\nEvaluating model...")

metrics = trainer.evaluate()

print("\nEvaluation metrics:")
print(metrics)


# ---------------------------------------
# 10. Save model
# ---------------------------------------

save_path = "./fine_tuned_distilbert_imdb"

model.save_pretrained(save_path)

tokenizer.save_pretrained(save_path)

print("\nModel saved successfully!")
print("Location:", save_path)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Using device: cuda

Loading IMDB dataset...
Training samples: 2000
Testing samples: 500

Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(



Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.351400,0.464328,0.822000
2,0.205000,0.604024,0.826000



Evaluating model...



Evaluation metrics:
{'eval_loss': 0.6040244102478027, 'eval_accuracy': 0.826, 'eval_runtime': 1.8719, 'eval_samples_per_second': 267.104, 'eval_steps_per_second': 33.655, 'epoch': 2.0}

Model saved successfully!
Location: ./fine_tuned_distilbert_imdb
